# 03 · Gold : modèle en étoile

Construit les tables que Power BI lira :

- **Faits :** `gold_fait_bons`, une ligne par bon.
- **Dimensions :** `gold_dim_date`, `gold_dim_pavillon`, `gold_dim_local`, `gold_dim_technicien`, `gold_dim_priorite`.
- **Tables d'analyse :** `gold_agg_mensuelle` et `gold_points_chauds`.

**Récidive :** un bon correctif non annulé, créé au plus 30 jours après le bon précédent du même local et de la même catégorie.

In [ ]:
FORMAT_TABLE = "delta"
JOURS_RECIDIVE = 30
SEUIL_POINT_CHAUD = 4

In [ ]:
from pyspark.sql import Window, functions as F

MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août",
        "septembre", "octobre", "novembre", "décembre"]
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]


def ecrire(df, nom):
    df.write.format(FORMAT_TABLE).mode("overwrite").option("overwriteSchema", "true").saveAsTable(nom)
    print(f"{nom} : {spark.table(nom).count()} lignes")


bons = spark.table("silver_bons_de_travail")

## Dimensions

`gold_dim_date` couvre des années complètes, comme l'exigent les fonctions de temps de DAX.

In [ ]:
bornes = bons.agg(F.year(F.min("date_creation")).alias("debut"), F.year(F.max("date_creation")).alias("fin")).first()
jour_semaine = (F.dayofweek("date") + 5) % 7 + 1  # lundi = 1

dim_date = (
    spark.sql(f"SELECT explode(sequence(DATE'{bornes.debut}-01-01', DATE'{bornes.fin}-12-31', INTERVAL 1 DAY)) AS date")
    .withColumn("annee", F.year("date"))
    .withColumn("trimestre", F.concat(F.lit("T"), F.quarter("date")))
    .withColumn("mois_numero", F.month("date"))
    .withColumn("mois_nom", F.element_at(F.array(*[F.lit(m) for m in MOIS]), F.month("date")))
    .withColumn("annee_mois", F.date_format("date", "yyyy-MM"))
    .withColumn("jour_semaine_numero", jour_semaine)
    .withColumn("jour_semaine_nom", F.element_at(F.array(*[F.lit(j) for j in JOURS]), jour_semaine))
    .withColumn("est_fin_de_semaine", (jour_semaine >= 6).cast("int"))
    .withColumn(
        "saison",
        F.when(F.month("date").isin(12, 1, 2), "Hiver")
        .when(F.month("date").isin(3, 4, 5), "Printemps")
        .when(F.month("date").isin(6, 7, 8), "Été")
        .otherwise("Automne"),
    )
)

techniciens = spark.table("silver_techniciens")
inconnu = spark.createDataFrame(
    [("INCONNU", "Inconnu", "Inconnu", "Inconnu", None)], schema=techniciens.schema,
)

ecrire(dim_date, "gold_dim_date")
ecrire(spark.table("silver_pavillons"), "gold_dim_pavillon")
ecrire(spark.table("silver_locaux"), "gold_dim_local")
ecrire(techniciens.unionByName(inconnu), "gold_dim_technicien")
ecrire(spark.table("silver_priorites"), "gold_dim_priorite")

## Table de faits

Les indicateurs valent 0 ou 1 : en DAX, un taux devient une simple division de deux sommes.

La fenêtre sépare les bons admissibles des autres, pour qu'un bon préventif ou annulé ne serve jamais de bon précédent.

In [ ]:
admissible = (F.col("type_bon") == "Correctif") & (F.col("statut") != "Annulé")
fenetre = Window.partitionBy("local_id", "categorie", "est_admissible_recidive").orderBy("date_creation", "bon_id")
ecart = F.col("date_creation").cast("long") - F.lag("date_creation").over(fenetre).cast("long")

fait = (
    bons
    .join(F.broadcast(spark.table("silver_priorites").select("priorite", "cible_fermeture_heures")), "priorite", "left")
    .withColumn("est_admissible_recidive", admissible.cast("int"))
    .withColumn(
        "est_recidive",
        F.when(admissible & (ecart <= JOURS_RECIDIVE * 86400), 1).otherwise(0),
    )
    .withColumn("est_ferme", (F.col("statut") == "Fermé").cast("int"))
    .withColumn("est_cible_respectee", F.when(
        F.col("date_fermeture").isNotNull(),
        (F.col("delai_fermeture_heures") <= F.col("cible_fermeture_heures")).cast("int"),
    ))
    .withColumn("date_creation_jour", F.to_date("date_creation"))
    .withColumn("date_fermeture_jour", F.to_date("date_fermeture"))
    .select(
        "bon_id", "bon_parent_id", "date_creation_jour", "date_fermeture_jour", "date_creation", "date_fermeture",
        "pavillon_id", "local_id", "technicien_id", "priorite", "statut", "type_bon", "categorie", "sous_categorie",
        "source", "description", "heures_main_oeuvre", "cout_main_oeuvre", "cout_pieces", "cout_sous_traitance",
        "cout_total", "cout_pieces_manquant", "delai_assignation_heures", "delai_fermeture_heures",
        "est_ferme", "est_cible_respectee", "est_admissible_recidive", "est_recidive",
    )
)
ecrire(fait, "gold_fait_bons")

## Tables d'analyse

`gold_agg_mensuelle` et `gold_points_chauds` donnent des réponses directes, utiles pour le rapport et pour l'agent IA de l'étape 6.

In [ ]:
fait = spark.table("gold_fait_bons")
actifs = fait.filter(F.col("statut") != "Annulé")

agg_mensuelle = (
    actifs
    .withColumn("mois", F.trunc("date_creation_jour", "month"))
    .groupBy("mois", "pavillon_id", "categorie")
    .agg(
        F.count("*").alias("nb_bons"),
        F.sum("est_ferme").alias("nb_fermes"),
        F.round(F.sum("cout_total"), 2).alias("cout_total"),
        F.avg("delai_fermeture_heures").alias("delai_moyen_heures"),
        F.sum("est_admissible_recidive").alias("nb_admissibles_recidive"),
        F.sum("est_recidive").alias("nb_recidives"),
    )
)

points_chauds = (
    fait.filter(F.col("est_admissible_recidive") == 1)
    .groupBy("local_id", "pavillon_id", "categorie")
    .agg(
        F.count("*").alias("nb_bons"),
        F.sum("est_recidive").alias("nb_recidives"),
        F.round(F.sum("cout_total"), 2).alias("cout_total"),
        F.min("date_creation_jour").alias("premier_bon"),
        F.max("date_creation_jour").alias("dernier_bon"),
    )
    .filter(F.col("nb_recidives") >= SEUIL_POINT_CHAUD)
    .join(spark.table("silver_locaux").select("local_id", "type_local", "criticite"), "local_id", "left")
)

ecrire(agg_mensuelle, "gold_agg_mensuelle")
ecrire(points_chauds, "gold_points_chauds")
display(spark.table("gold_points_chauds").orderBy(F.desc("nb_recidives")).limit(20))

## Contrôles

Chaque clé de la table de faits doit exister dans sa dimension, sinon Power BI regrouperait ces bons sous « (vide) ».

In [ ]:
cles = [
    ("date_creation_jour", "gold_dim_date", "date"),
    ("pavillon_id", "gold_dim_pavillon", "pavillon_id"),
    ("local_id", "gold_dim_local", "local_id"),
    ("technicien_id", "gold_dim_technicien", "technicien_id"),
    ("priorite", "gold_dim_priorite", "priorite"),
]
for cle, dimension, colonne in cles:
    valeurs = spark.table(dimension).select(F.col(colonne).alias(cle))
    orphelins = fait.filter(F.col(cle).isNotNull()).join(valeurs, cle, "left_anti").count()
    assert orphelins == 0, f"{orphelins} clés {cle} absentes de {dimension}"

assert fait.count() == bons.count(), "la table de faits n'a pas le même nombre de lignes que silver"
assert spark.table("gold_dim_local").groupBy("local_id").count().filter("count > 1").count() == 0
print("Contrôles réussis")